In [ ]:
# 03 – Feature Engineering (Batting)

Purpose:
- Compute batter-level performance metrics
- Compute phase-wise participation metrics
- Create final batting feature dataset

Input:
- data/processed/deliveries_cleaned.csv

Output:
- data/processed/batting_features_final_v2.csv

In [3]:
import sys
from pathlib import Path
sys.path.append(str(Path.cwd().parent))

from config import *
import pandas as pd
import numpy as np

In [4]:
df = pd.read_csv(PROCESSED_DIR / "deliveries_cleaned.csv")

print(df.shape)
df.head()

(278205, 15)


,match_id,innings,batting_team,over,ball,batsman,bowler,non_striker,runs_batsman,runs_extras,runs_total,extras_type,player_dismissed,dismissal_kind,phase
0,1,1st innings,Sunrisers Hyderabad,0,1,DA Warner,TS Mills,S Dhawan,0,0,0,NaN,NaN,NaN,Powerplay
1,1,1st innings,Sunrisers Hyderabad,0,2,DA Warner,TS Mills,S Dhawan,0,0,0,NaN,NaN,NaN,Powerplay
2,1,1st innings,Sunrisers Hyderabad,0,3,DA Warner,TS Mills,S Dhawan,4,0,4,NaN,NaN,NaN,Powerplay
3,1,1st innings,Sunrisers Hyderabad,0,4,DA Warner,TS Mills,S Dhawan,0,0,0,NaN,NaN,NaN,Powerplay
4,1,1st innings,Sunrisers Hyderabad,0,5,DA Warner,TS Mills,S Dhawan,0,2,2,wides,NaN,NaN,Powerplay


In [7]:
df["balls_faced"] = 1
batting_basic = (
    df.groupby("batsman")
      .agg(
          runs=("runs_batsman", "sum"),
          balls=("balls_faced", "sum"),
          dismissals=("player_dismissed", lambda x: x.notna().sum())
      )
      .reset_index()
)

batting_basic["strike_rate"] = (
    batting_basic["runs"] / batting_basic["balls"] * 100
)

batting_basic["average"] = (
    batting_basic["runs"] / batting_basic["dismissals"]
)

batting_basic.head()

,batsman,runs,balls,dismissals,strike_rate,average
0,A Ashish Reddy,280,196,15,142.857143,18.666667
1,A Badoni,963,740,36,130.135135,26.750000
2,A Chandila,4,7,1,57.142857,4.000000
3,A Chopra,53,75,5,70.666667,10.600000
4,A Choudhary,25,20,2,125.000000,12.500000


In [8]:
matches_played = (
    df.groupby("batsman")["match_id"]
      .nunique()
      .reset_index(name="matches_played")
)

In [9]:
phase_sr = (
    df.groupby(["batsman", "phase"])
      .agg(
          runs=("runs_batsman", "sum"),
          balls=("balls_faced", "sum")
      )
      .reset_index()
)

phase_sr["strike_rate"] = phase_sr["runs"] / phase_sr["balls"] * 100

phase_sr_pivot = (
    phase_sr
    .pivot(index="batsman", columns="phase", values="strike_rate")
    .reset_index()
)

In [10]:
df["boundary_runs"] = np.where(
    df["runs_batsman"].isin([4, 6]),
    df["runs_batsman"],
    0
)

boundary_pct = (
    df.groupby("batsman")
      .agg(
          total_runs=("runs_batsman", "sum"),
          boundary_runs=("boundary_runs", "sum")
      )
      .reset_index()
)

boundary_pct["boundary_pct"] = (
    boundary_pct["boundary_runs"] / boundary_pct["total_runs"] * 100
)

boundary_pct = boundary_pct[["batsman", "boundary_pct"]]

In [11]:
df["is_dot_ball"] = (
    (df["runs_batsman"] == 0) &
    (df["balls_faced"] == 1)
).astype(int)

dot_ball_pct = (
    df.groupby("batsman")
      .agg(
          dot_balls=("is_dot_ball", "sum"),
          balls_faced=("balls_faced", "sum")
      )
      .reset_index()
)

dot_ball_pct["dot_ball_pct"] = (
    dot_ball_pct["dot_balls"] / dot_ball_pct["balls_faced"] * 100
)

dot_ball_pct = dot_ball_pct[["batsman", "dot_ball_pct"]]

In [12]:
df["is_dismissed"] = df["player_dismissed"].notna().astype(int)

dismissal_rate = (
    df.groupby("batsman")
      .agg(
          balls_faced=("balls_faced", "sum"),
          dismissals=("is_dismissed", "sum")
      )
      .reset_index()
)

dismissal_rate["dismissal_rate"] = (
    dismissal_rate["balls_faced"] / dismissal_rate["dismissals"]
)

dismissal_rate = dismissal_rate[["batsman", "dismissal_rate"]]

In [13]:
matches = (
    df.groupby("batsman")["match_id"]
      .nunique()
      .reset_index(name="matches_played")
)

pp_balls = (
    df[df["phase"] == "Powerplay"]
    .groupby("batsman")["balls_faced"]
    .sum()
    .reset_index(name="pp_balls")
)

middle_balls = (
    df[df["phase"] == "Middle"]
    .groupby("batsman")["balls_faced"]
    .sum()
    .reset_index(name="middle_balls")
)

death_balls = (
    df[df["phase"] == "Death"]
    .groupby("batsman")["balls_faced"]
    .sum()
    .reset_index(name="death_balls")
)

In [14]:
phase_usage = (
    matches
    .merge(pp_balls, on="batsman", how="left")
    .merge(middle_balls, on="batsman", how="left")
    .merge(death_balls, on="batsman", how="left")
)

phase_usage.fillna(0, inplace=True)

phase_usage["avg_pp_balls_per_match"] = (
    phase_usage["pp_balls"] / phase_usage["matches_played"]
)

phase_usage["avg_middle_balls_per_match"] = (
    phase_usage["middle_balls"] / phase_usage["matches_played"]
)

phase_usage["avg_death_balls_per_match"] = (
    phase_usage["death_balls"] / phase_usage["matches_played"]
)

In [15]:
batting_features = (
    batting_basic
    .merge(matches_played, on="batsman", how="left")
    .merge(phase_sr_pivot, on="batsman", how="left")
    .merge(boundary_pct, on="batsman", how="left")
    .merge(dot_ball_pct, on="batsman", how="left")
    .merge(dismissal_rate, on="batsman", how="left")
    .merge(phase_usage[
        ["batsman",
         "avg_pp_balls_per_match",
         "avg_middle_balls_per_match",
         "avg_death_balls_per_match"]
    ], on="batsman", how="left")
)

In [16]:
batting_features.to_csv(
    PROCESSED_DIR / "batting_features_final_v2.csv",
    index=False
)

print("batting_features_final_v2.csv created successfully")

batting_features_final_v2.csv created successfully
